# SDE and Probability Flow ODE Match

Chapter 21: Fokker-Planck and Distribution Dynamics · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare distribution evolution under stochastic SDE and deterministic probability-flow ODE.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-21/sde-ode-match/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(0)

# The demo's defaults: OU SDE dX = -lam X dt + sig dW from X0 ~ N(0, var0), up to time T
lam, sig, var0, T = 1.0, 0.55, 1.6, 2.2
n_steps, n = 2200, 50000
h = T / n_steps


def v(t):
    """Variance of the SDE marginal p_t = N(0, v(t))."""
    return var0 * math.exp(-2 * lam * t) + sig ** 2 * (1 - math.exp(-2 * lam * t)) / (2 * lam)


x_sde = math.sqrt(var0) * torch.randn(n, dtype=torch.float64)
x_pf = x_sde.clone()          # the same starting points
x_drift = x_sde.clone()
t = 0.0
for _ in range(n_steps):
    x_sde = x_sde - lam * x_sde * h + sig * math.sqrt(h) * torch.randn_like(x_sde)
    score = -x_pf / v(t)                                   # d/dx log p_t
    x_pf = x_pf + (-lam * x_pf - 0.5 * sig ** 2 * score) * h   # probability-flow ODE: no noise
    x_drift = x_drift - lam * x_drift * h                      # drift only: score term dropped
    t += h

print(f"variance at T = {T}:")
print(f"  SDE                   {x_sde.var().item():.4f}")
print(f"  probability-flow ODE  {x_pf.var().item():.4f}")
print(f"  exact v(T)            {v(T):.4f}")
print(f"  drift-only ODE        {x_drift.var().item():.4f}   (= var0 e^(-2 lam T) = {var0 * math.exp(-2 * lam * T):.4f}: does not match)")

# Same marginals, different paths: the ODE map is deterministic and keeps the order of points
start = math.sqrt(var0) * torch.randn(5, dtype=torch.float64)
y, t = start.clone(), 0.0
for _ in range(n_steps):
    y = y + (-lam * y + 0.5 * sig ** 2 * y / v(t)) * h
    t += h
print("\nPF-ODE keeps the order of points:", torch.equal(start.argsort(), y.argsort()),
      "| it scales every point by", round((y / start).mean().item(), 4), "= sqrt(v(T)/var0) =", round(math.sqrt(v(T) / var0), 4))

variance at T = 2.2:
  SDE                   0.1693
  probability-flow ODE  0.1698
  exact v(T)            0.1690
  drift-only ODE        0.0197   (= var0 e^(-2 lam T) = 0.0196: does not match)

PF-ODE keeps the order of points: True | it scales every point by 0.3248 = sqrt(v(T)/var0) = 0.325
